[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/testing_ml_code.ipynb)

# 06 . Testing ML Code

## Concept — Testing ML Code (revision notes)

**What it is**
- Automated checks that catch the bugs ML code is especially prone to: silent shape errors, broken gradient flow, nondeterminism, data leakage and models that train but learn nothing.
- Tests here are ordinary pytest/unittest tests: small, fast, deterministic, run on every commit. We run them in the notebook with `unittest`; the same bodies work as pytest functions.

**Why it matters**
- ML bugs rarely crash. A detached tensor, a wrong `dim=` or a missing `zero_grad()` gives a model that trains to a mediocre result, and you lose days finding it.
- Cheap tests (milliseconds, tiny models, synthetic data) catch most of these before an expensive training run.

**When to use**
- Unit-test every custom layer, loss, metric, transform and data-collate function.
- Add smoke tests for the training step (loss goes down on one batch).
- Keep expensive end-to-end / accuracy tests separate (marked slow, run nightly).

**Math & intuition**
- Shape tests: output shape for several batch sizes (including batch size 1).
- Overfit-one-batch: a correct model + optimizer must drive loss near zero on one small batch; if not, something is broken in model, loss, labels or optimizer.
- Gradient flow: after `backward()` every trainable parameter should have a finite, non-zero `.grad`.
- Determinism: same seed -> same output; eval mode -> dropout off; batch independence (row i's output must not depend on other rows in eval mode).
- Initial loss sanity: for K balanced classes with small init, loss should be about ln(K).

### 1. The code under test

**What this cell does (plain language):** we define a small classifier, a custom label-smoothing loss and a `normalize` preprocessing function. These are the kinds of components you would normally import from your package; we define them here so the notebook is self-contained.

In [1]:
import math, unittest, io, sys, copy
import torch, torch.nn as nn, torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, d_in=8, d_hidden=32, n_classes=3, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(d_hidden, n_classes))
    def forward(self, x): return self.net(x)

def smoothed_ce(logits, target, eps=0.1):
    # label-smoothed cross entropy: (1-eps)*NLL + eps*uniform
    logp = F.log_softmax(logits, dim=-1)
    nll = -logp.gather(1, target[:, None]).squeeze(1)
    uniform = -logp.mean(dim=-1)
    return ((1 - eps) * nll + eps * uniform).mean()

def normalize(x, eps=1e-6):
    # standardize each row to zero mean / unit variance
    return (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True, unbiased=False) + eps)

print("defined MLP, smoothed_ce, normalize")

defined MLP, smoothed_ce, normalize


### 2. Shape and basic-output tests

**What this cell does (plain language):** the first test class checks output shapes for several batch sizes (using `subTest` so each failing size is reported separately), that outputs are finite, and that `normalize` really produces zero mean / unit std rows. We define the classes now and run them all together in section 8.

In [2]:
class TestShapes(unittest.TestCase):
    def test_output_shape_various_batch_sizes(self):
        model = MLP().eval()
        for bs in (1, 2, 17):                       # batch size 1 is a classic edge case
            with self.subTest(batch=bs):
                self.assertEqual(model(torch.randn(bs, 8)).shape, (bs, 3))

    def test_output_is_finite(self):
        out = MLP().eval()(torch.randn(5, 8))
        self.assertTrue(torch.isfinite(out).all())

    def test_normalize_stats(self):
        z = normalize(torch.randn(6, 20) * 5 + 3)
        torch.testing.assert_close(z.mean(-1), torch.zeros(6), atol=1e-5, rtol=0)
        torch.testing.assert_close(z.std(-1, unbiased=False), torch.ones(6), atol=1e-4, rtol=0)

    def test_wrong_input_dim_raises(self):
        with self.assertRaises(RuntimeError):
            MLP()(torch.randn(4, 7))

### 3. Overfit-one-batch test

**What this cell does (plain language):** the single most valuable ML test: train on ONE tiny batch for a few hundred steps and assert the loss collapses. A healthy model/loss/optimizer combination memorizes 16 examples easily. If this fails, no amount of data or tuning will save you. We make it quick (fixed seed, small model, early stop at a loss threshold).

In [3]:
def overfit_one_batch(model, loss_fn, steps=300, lr=0.05, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(16, 8, generator=g); y = torch.randint(0, 3, (16,), generator=g)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    first = None
    for _ in range(steps):
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        first = first if first is not None else loss.item()
        if loss.item() < 0.05: break
    return first, loss.item()

class TestOverfit(unittest.TestCase):
    def test_can_overfit_one_batch(self):
        torch.manual_seed(0)
        first, last = overfit_one_batch(MLP(dropout=0.0), F.cross_entropy)
        self.assertLess(last, 0.1, f"loss only went {first:.3f} -> {last:.3f}")

    def test_initial_loss_is_ln_k(self):
        torch.manual_seed(0)
        model = MLP(); model.eval()
        x = torch.randn(256, 8); y = torch.randint(0, 3, (256,))
        loss = F.cross_entropy(model(x), y).item()
        self.assertAlmostEqual(loss, math.log(3), delta=0.4)    # ~1.10 for 3 classes

### 4. Determinism and eval-mode tests

**What this cell does (plain language):** we test that the same seed gives bit-identical outputs and training results, that two forward passes in eval mode agree (dropout off), and that in train mode dropout makes them differ. The last check proves the test can fail: it asserts the nondeterminism that dropout is supposed to introduce.

In [4]:
class TestDeterminism(unittest.TestCase):
    def test_same_seed_same_init(self):
        torch.manual_seed(5); a = MLP()
        torch.manual_seed(5); b = MLP()
        for pa, pb in zip(a.parameters(), b.parameters()):
            self.assertTrue(torch.equal(pa, pb))

    def test_same_seed_same_training(self):
        r1 = overfit_one_batch((torch.manual_seed(1), MLP())[1], F.cross_entropy, steps=20)
        r2 = overfit_one_batch((torch.manual_seed(1), MLP())[1], F.cross_entropy, steps=20)
        self.assertEqual(r1, r2)

    def test_eval_is_deterministic(self):
        m = MLP(dropout=0.5).eval(); x = torch.randn(4, 8)
        self.assertTrue(torch.equal(m(x), m(x)))

    def test_train_mode_dropout_is_stochastic(self):
        m = MLP(dropout=0.5).train(); x = torch.randn(4, 8)
        self.assertFalse(torch.equal(m(x), m(x)))

### 5. Gradient-flow tests

**What this cell does (plain language):** we check that after one backward pass every trainable parameter received a gradient, that gradients are finite and not all zero, and that one optimizer step actually changes the weights. Then we define a deliberately buggy model (it calls `.detach()` in the middle, a classic mistake) and assert that our test detects it — a test of the test.

In [5]:
class DetachedMLP(MLP):
    def forward(self, x):
        h = self.net[0](x).detach()          # BUG: cuts the graph, first layer never gets gradients
        return self.net[3](F.relu(h))

def grads_of(model):
    model.zero_grad()
    smoothed_ce(model(torch.randn(8, 8)), torch.randint(0, 3, (8,))).backward()
    return {n: p.grad for n, p in model.named_parameters()}

class TestGradients(unittest.TestCase):
    def test_all_params_get_finite_nonzero_grads(self):
        torch.manual_seed(0)
        for name, g in grads_of(MLP(dropout=0.0)).items():
            with self.subTest(param=name):
                self.assertIsNotNone(g, "no gradient (detached or unused?)")
                self.assertTrue(torch.isfinite(g).all())
                self.assertGreater(g.abs().sum().item(), 0.0)

    def test_step_changes_weights(self):
        torch.manual_seed(0); m = MLP(); before = copy.deepcopy(m.state_dict())
        opt = torch.optim.SGD(m.parameters(), lr=0.1)
        grads_of(m); opt.step()
        changed = [not torch.equal(before[k], v) for k, v in m.state_dict().items()]
        self.assertTrue(all(changed))

    def test_detects_detach_bug(self):
        torch.manual_seed(0)
        grads = grads_of(DetachedMLP(dropout=0.0))
        self.assertIsNone(grads["net.0.weight"])        # the bug really is detected

### 6. Numerical correctness: reference implementations and gradcheck

**What this cell does (plain language):** custom losses and ops should be checked against a trusted reference. We compare `smoothed_ce` with PyTorch's built-in `F.cross_entropy(label_smoothing=...)`, test the `eps=0` special case equals plain cross-entropy, and run `torch.autograd.gradcheck` (finite differences, needs float64) on the loss.

In [6]:
class TestNumerics(unittest.TestCase):
    def test_matches_builtin_label_smoothing(self):
        torch.manual_seed(0)
        logits = torch.randn(10, 5); y = torch.randint(0, 5, (10,))
        torch.testing.assert_close(smoothed_ce(logits, y, 0.1), F.cross_entropy(logits, y, label_smoothing=0.1))

    def test_eps_zero_is_plain_ce(self):
        torch.manual_seed(1)
        logits = torch.randn(10, 5); y = torch.randint(0, 5, (10,))
        torch.testing.assert_close(smoothed_ce(logits, y, 0.0), F.cross_entropy(logits, y))

    def test_gradcheck(self):
        torch.manual_seed(2)
        logits = torch.randn(4, 3, dtype=torch.float64, requires_grad=True)   # float64 is required
        y = torch.randint(0, 3, (4,))
        self.assertTrue(torch.autograd.gradcheck(lambda z: smoothed_ce(z, y), (logits,)))

    def test_stable_for_large_logits(self):
        v = smoothed_ce(torch.tensor([[1000.0, -1000.0, 0.0]]), torch.tensor([0]))
        self.assertTrue(torch.isfinite(v))

### 7. Invariance and property tests

**What this cell does (plain language):** these tests encode properties that must hold regardless of the specific numbers: in eval mode each row's output must not depend on the other rows in the batch (a classic bug is a layer that mixes samples, e.g. normalizing over the batch dim), and permuting the batch must permute the outputs the same way. We also show a buggy `BatchMixer` module failing the independence test.

In [7]:
class BatchMixer(nn.Module):
    def forward(self, x): return x - x.mean(0, keepdim=True)    # BUG: statistics across the batch

def batch_independent(model, atol=1e-6):
    model.eval(); x = torch.randn(6, 8)
    full = model(x)
    single = torch.cat([model(x[i:i + 1]) for i in range(6)])    # one at a time
    return torch.allclose(full, single, atol=atol)

class TestInvariants(unittest.TestCase):
    def test_batch_independence(self):
        torch.manual_seed(0)
        self.assertTrue(batch_independent(MLP()))

    def test_permutation_equivariance(self):
        torch.manual_seed(0); m = MLP().eval(); x = torch.randn(6, 8); perm = torch.randperm(6)
        torch.testing.assert_close(m(x)[perm], m(x[perm]))

    def test_batch_mixer_is_caught(self):
        torch.manual_seed(0)
        self.assertFalse(batch_independent(BatchMixer()))   # confirms the property test has teeth

### 8. Running the whole suite with unittest inside the notebook

**What this cell does (plain language):** we collect every test class defined above into one suite and run it with `unittest.TextTestRunner`, writing to the notebook's stdout. The same classes can be dropped into `tests/test_model.py` and run with `python -m unittest` or `pytest`. The cell asserts that everything passed so a broken test fails the notebook loudly.

In [8]:
loader = unittest.TestLoader()
suite = unittest.TestSuite(loader.loadTestsFromTestCase(c) for c in
                           (TestShapes, TestOverfit, TestDeterminism, TestGradients, TestNumerics, TestInvariants))
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
print("tests run:", result.testsRun, "| failures:", len(result.failures), "| errors:", len(result.errors))
assert result.wasSuccessful()

test_normalize_stats (__main__.TestShapes.test_normalize_stats) ... 

ok


test_output_is_finite (__main__.TestShapes.test_output_is_finite) ... 

ok


test_output_shape_various_batch_sizes (__main__.TestShapes.test_output_shape_various_batch_sizes) ... 

...

ok


test_permutation_equivariance (__main__.TestInvariants.test_permutation_equivariance) ... 

ok


----------------------------------------------------------------------
Ran 20 tests in 1.672s

OK


tests run: 20 | failures: 0 | errors: 0


### 9. pytest style and a mutation check (guarded)

**What this cell does (plain language):** in a real repo you would write plain functions with `assert` and `pytest.mark.parametrize`; we show that form below as text. We then run a mini 'mutation test': break the loss on purpose (swap the sign) and confirm that at least one of our tests fails. A test suite that cannot fail is worthless. If `pytest` is installed we just report its version.

In [9]:
pytest_style = '''
# tests/test_model.py  (pytest style)
import pytest, torch
@pytest.mark.parametrize("bs", [1, 2, 17])
def test_shapes(bs):
    assert MLP().eval()(torch.randn(bs, 8)).shape == (bs, 3)

def test_overfit():
    assert overfit_one_batch(MLP(dropout=0.0), F.cross_entropy)[1] < 0.1
'''
print(pytest_style)
try:
    import pytest; print("pytest available:", pytest.__version__)
except ImportError:
    print("pytest not installed -> `pip install pytest`; unittest above works without it")

# Mutation check: sabotage the loss and see whether the suite notices.
_orig = smoothed_ce
def smoothed_ce(logits, target, eps=0.1): return -_orig(logits, target, eps)     # BUG: sign flipped
res = unittest.TextTestRunner(stream=io.StringIO()).run(loader.loadTestsFromTestCase(TestNumerics))
print("mutant caught (suite failed):", not res.wasSuccessful(), "| failing tests:", len(res.failures) + len(res.errors))
smoothed_ce = _orig                                                               # restore


# tests/test_model.py  (pytest style)
import pytest, torch
@pytest.mark.parametrize("bs", [1, 2, 17])
def test_shapes(bs):
    assert MLP().eval()(torch.randn(bs, 8)).shape == (bs, 3)

def test_overfit():
    assert overfit_one_batch(MLP(dropout=0.0), F.cross_entropy)[1] < 0.1

pytest not installed -> `pip install pytest`; unittest above works without it
mutant caught (suite failed): True | failing tests: 2


### 10. Data tests: shape contracts and leakage checks

**What this cell does (plain language):** data bugs are as common as model bugs. We test a train/val split function: sizes add up, no index appears in both splits (leakage), and the split is reproducible with a seed. We also check that labels are in the valid range [0, K).

In [10]:
def split_indices(n, val_frac, seed):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g).tolist()
    k = int(n * val_frac)
    return perm[k:], perm[:k]

class TestData(unittest.TestCase):
    def test_split_sizes_and_disjoint(self):
        tr, va = split_indices(100, 0.2, seed=0)
        self.assertEqual((len(tr), len(va)), (80, 20))
        self.assertFalse(set(tr) & set(va), "train/val leakage!")
        self.assertEqual(sorted(tr + va), list(range(100)))

    def test_split_reproducible(self):
        self.assertEqual(split_indices(50, 0.3, 7), split_indices(50, 0.3, 7))

    def test_labels_in_range(self):
        y = torch.randint(0, 3, (100,))
        self.assertTrue(((y >= 0) & (y < 3)).all())

r = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(loader.loadTestsFromTestCase(TestData))
assert r.wasSuccessful()

.

.

.

----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


## Common bugs

- **Tests that depend on unseeded randomness (flaky tests).** Fix: Seed in every test (`torch.manual_seed`) and use tolerances rather than exact floats where appropriate.
- **Comparing floats with `==`.** Fix: Use `torch.testing.assert_close` / `assertAlmostEqual` with explicit `atol`/`rtol`.
- **Testing only batch size 1 or only the example batch size.** Fix: Parametrize over batch sizes including 1 and odd numbers; many bugs show up only at batch 1 (squeeze) or when batch != a multiple of something.
- **Forgetting `model.eval()` in deterministic/equality tests.** Fix: Dropout and batch norm make train mode stochastic/batch-dependent; set the mode explicitly in each test.
- **Overfit test that is too slow or too strict.** Fix: Use a tiny model, 16 samples, fixed seed, early stop, and a loose threshold (loss < 0.1), not a perfect 0.
- **Checking only that loss decreases on real data.** Fix: Also check gradient flow and parameter updates; a loss can decrease from a bias term alone while most of the network is detached.
- **`gradcheck` with float32 inputs.** Fix: Use float64 and `requires_grad=True`; float32 gives spurious failures.
- **Tests that need a GPU or network to run.** Fix: Use CPU, synthetic data and tiny shapes; mark any heavy test `slow` and run it separately.
- **Never watching a test fail.** Fix: Do a mutation check (break the code on purpose) once to prove the test has teeth.

## Exercises

**Exercise 1.** Write a test case `TestNoNaN` asserting that the output of `MLP` is finite even for an input with large values (e.g. `1e4 * randn`). Run it with `unittest`.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
class TestNoNaN(unittest.TestCase):
    def test_large_inputs(self):
        torch.manual_seed(0)
        out = MLP().eval()(1e4 * torch.randn(4, 8))
        self.assertTrue(torch.isfinite(out).all())
unittest.TextTestRunner(stream=sys.stdout).run(unittest.TestLoader().loadTestsFromTestCase(TestNoNaN))

.

----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

**Exercise 2.** Write a test that every parameter of `MLP` has `requires_grad=True`, and one that freezing the first layer leaves its weights unchanged after an optimizer step.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
torch.manual_seed(0)
m = MLP(dropout=0.0)
assert all(p.requires_grad for p in m.parameters())
for p in m.net[0].parameters(): p.requires_grad_(False)
w0 = m.net[0].weight.clone()
opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=0.1)
smoothed_ce(m(torch.randn(8, 8)), torch.randint(0, 3, (8,))).backward(); opt.step()
print("frozen layer unchanged:", torch.equal(w0, m.net[0].weight))

frozen layer unchanged: True


**Exercise 3.** Add a test that a model's `state_dict` round-trips: save to a `BytesIO`, load into a fresh model and check outputs match in eval mode.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
torch.manual_seed(0)
a = MLP().eval(); buf = io.BytesIO(); torch.save(a.state_dict(), buf); buf.seek(0)
b = MLP().eval(); b.load_state_dict(torch.load(buf))
x = torch.randn(3, 8)
torch.testing.assert_close(a(x), b(x)); print("round trip OK")

round trip OK


**Exercise 4.** Write a property test: `normalize(c * x)` should equal `normalize(x)` for any positive scalar c (scale invariance). Check c = 1, 10, 0.01.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

The eps in the denominator breaks exact invariance for tiny inputs, hence the tolerance.

In [18]:
# Solution 4
torch.manual_seed(0); x = torch.randn(5, 12)
for c in (1.0, 10.0, 0.01):
    torch.testing.assert_close(normalize(c * x), normalize(x), atol=2e-3, rtol=1e-3)
print("scale invariance holds")

scale invariance holds


**Exercise 5.** Use `subTest` to parametrize a test over `n_classes` in (2, 3, 10): the initial loss should be roughly ln(n_classes).

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
class TestInitLoss(unittest.TestCase):
    def test_ln_k(self):
        for k in (2, 3, 10):
            with self.subTest(k=k):
                torch.manual_seed(0)
                m = MLP(n_classes=k).eval()
                loss = F.cross_entropy(m(torch.randn(512, 8)), torch.randint(0, k, (512,))).item()
                self.assertAlmostEqual(loss, math.log(k), delta=0.5)
r = unittest.TextTestRunner(stream=sys.stdout).run(unittest.TestLoader().loadTestsFromTestCase(TestInitLoss))

.

----------------------------------------------------------------------
Ran 1 test in 0.009s

OK
